# Parte 3: Análisis CLTV (Customer Lifetime Value)

Comparación de estrategias de acción comercial basadas en:
- **Parte 2**: Segmentación solo por riesgo de churn (5 grupos)
- **Parte 3**: Segmentación por riesgo de churn × valor del cliente (CLTV) - 15 estrategias personalizadas

Objetivo: Validar si la personalización adicional por CLTV mejora el ROI global.

In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:,.2f}'.format

# Cargar datos
dm = pd.read_csv('Data/Datamart/datamart_final_v2.csv')
df_nuevos = pd.read_csv('Data/DataLake/nuevos_clientes.csv')
df_costes = pd.read_csv('Data/DataLake/Costes.csv')

print(f'Datos entrenamiento: {dm.shape}')
print(f'Nuevos clientes: {df_nuevos.shape}')
print(f'Costes modelos: {df_costes.shape}')

Datos entrenamiento: (58049, 26)
Nuevos clientes: (10000, 38)
Costes modelos: (11, 7)


## 1. Preparación de Features

In [13]:
# Define features (mismo que Enfoque 2)
FEATURES_NUMERICAS = ['km_ultima_revision', 'PVP', 'Edad', 'RENTA_MEDIA_ESTIMADA', 'gasto_relativo', 'Kw', 'Margen_eur_bruto', 'Margen_eur', 'ENCUESTA_CLIENTE_ZONA_TALLER']

FEATURES_BINARIAS = ['tiene_queja', 'en_garantia_bin', 'MANTENIMIENTO_GRATUITO', 'Lead_compra', 'seguro_bateria_bin', 'sin_encuesta', 'origen_internet']

FEATURES_CATEGORICAS = ['Modelo', 'ZONA', 'FORMA_PAGO', 'TIPO_CARROCERIA', 'Equipamiento', 'Fuel', 'TRANSMISION_ID']

ALL_FEATURES = FEATURES_NUMERICAS + FEATURES_BINARIAS + FEATURES_CATEGORICAS

# Preparar datos de entrenamiento
X_train = dm[ALL_FEATURES].copy()
y_train = dm['Churn_Corregido'].copy()

# Codificar variables categóricas para entrenamiento
label_encoders = {}
for col in FEATURES_CATEGORICAS:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    label_encoders[col] = le

print('Features preparados')

Features preparados


## 2. Entrenamiento XGBoost

In [14]:
# Entrenar XGBoost (Enfoque 2)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=2,
    random_state=42
)

xgb_model.fit(X_train, y_train)
print('Modelo XGBoost entrenado')

Modelo XGBoost entrenado


## 3. Feature Engineering para Nuevos Clientes

In [15]:
# Crear features faltantes en nuevos_clientes
fresh = df_nuevos.copy()

# Convertir QUEJA (es numérico) a binario
fresh['tiene_queja'] = (fresh['QUEJA'].fillna(0).astype(int) == 1).astype(int)

# Convertir EN_GARANTIA a binario
fresh['en_garantia_bin'] = (fresh['EN_GARANTIA'].fillna('NO').astype(str).str.upper().str.strip() == 'SI').astype(int)

# Crear gasto relativo
fresh['gasto_relativo'] = fresh['PVP'] / fresh['RENTA_MEDIA_ESTIMADA'].clip(lower=1)

# Seguro batería
fresh['seguro_bateria_bin'] = (fresh['SEGURO_BATERIA_LARGO_PLAZO'].fillna('NO').astype(str).str.upper().str.strip() == 'SI').astype(int)

# Sin encuesta
fresh['sin_encuesta'] = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].isnull().astype(int)
fresh['ENCUESTA_CLIENTE_ZONA_TALLER'] = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].fillna(0)

# Origen internet
fresh['origen_internet'] = (fresh['Origen'].fillna('Tienda').astype(str).str.strip() == 'Internet').astype(int)

# Copiar features
fresh_feat = fresh[ALL_FEATURES].copy()

# Rellenar valores faltantes (numéricas con mediana)
for col in FEATURES_NUMERICAS:
    if col == 'ENCUESTA_CLIENTE_ZONA_TALLER':
        continue
    if fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].median())

# Codificar categóricas con label encoders del entrenamiento
for col in FEATURES_CATEGORICAS:
    if fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].mode()[0])
    
    le = label_encoders[col]
    known_classes = set(le.classes_)
    
    # Convertir valores desconocidos a la primera clase conocida
    def encode_value(x):
        x_str = str(x)
        if x_str in known_classes:
            return le.transform([x_str])[0]
        else:
            return le.transform([le.classes_[0]])[0]
    
    fresh_feat[col] = fresh_feat[col].astype(str).apply(encode_value)

# Alinear con X_train
fresh_aligned = fresh_feat.reindex(columns=X_train.columns, fill_value=0)

print('Features de nuevos clientes preparados')

Features de nuevos clientes preparados


## 4. Predicción de Churn

In [16]:
# Predecir probabilidades de churn
probs_churn = xgb_model.predict_proba(fresh_aligned)[:, 1]
fresh['prob_churn'] = probs_churn

# Segmentar por churn
def segmento_churn(prob):
    if prob >= 0.80: return 'MUY_ALTO'
    elif prob >= 0.60: return 'ALTO'
    elif prob >= 0.40: return 'MEDIO'
    elif prob >= 0.20: return 'BAJO'
    else: return 'MUY_BAJO'

fresh['segmento_churn'] = fresh['prob_churn'].apply(segmento_churn)

print(f'Distribucion de riesgo:')
print(fresh['segmento_churn'].value_counts().sort_index())

Distribucion de riesgo:
segmento_churn
ALTO        5821
BAJO          24
MEDIO       2163
MUY_ALTO     531
MUY_BAJO    1461
Name: count, dtype: int64


## 5. Cálculo de CLTV (Customer Lifetime Value)

In [17]:
# Parámetros CLTV
HORIZONTE_ANOS = 5
TASA_DESCUENTO = 0.10
TASAS_RETENCION_BASE = {
    'MUY_ALTO': 0.50,
    'ALTO': 0.65,
    'MEDIO': 0.78,
    'BAJO': 0.88,
    'MUY_BAJO': 0.95
}

# Usar el Margen_eur_bruto que ya existe en fresh (viene del csv)
# Si no existe, calcularlo a partir del PVP y un margen base del 28%
if 'Margen_eur_bruto' not in fresh.columns or fresh['Margen_eur_bruto'].isnull().all():
    fresh['Margen_eur_bruto'] = fresh['PVP'] * 0.28

def calcular_cltv(row):
    margen_por_visit = row['Margen_eur_bruto']  # Margen en EUR por venta/revisión
    tasa_retencion = TASAS_RETENCION_BASE[row['segmento_churn']]
    
    cltv = 0
    prob_activo = 1.0
    
    for ano in range(1, HORIZONTE_ANOS + 1):
        flujo = margen_por_visit * prob_activo
        cltv += flujo / ((1 + TASA_DESCUENTO) ** ano)
        prob_activo *= tasa_retencion
    
    return cltv

fresh['CLTV'] = fresh.apply(calcular_cltv, axis=1)

# Segmentar por CLTV (terciles)
p33 = fresh['CLTV'].quantile(0.33)
p67 = fresh['CLTV'].quantile(0.67)

def segmento_cltv(valor):
    if valor >= p67: return 'ALTO'
    elif valor >= p33: return 'MEDIO'
    else: return 'BAJO'

fresh['segmento_cltv'] = fresh['CLTV'].apply(segmento_cltv)

print(f'Estadisticas CLTV:')
print(f'  Media: {fresh["CLTV"].mean():.2f} EUR')
print(f'  Mediana: {fresh["CLTV"].median():.2f} EUR')
print(f'  Min-Max: {fresh["CLTV"].min():.2f} EUR - {fresh["CLTV"].max():.2f} EUR')
print(f'\nDistribucion CLTV:')
print(fresh['segmento_cltv'].value_counts())

Estadisticas CLTV:
  Media: 15002.97 EUR
  Mediana: 14609.75 EUR
  Min-Max: 2463.66 EUR - 43264.45 EUR

Distribucion CLTV:
segmento_cltv
MEDIO    3400
ALTO     3302
BAJO     3298
Name: count, dtype: int64


## 6. Matriz de Estrategias 2D (Churn × CLTV)

In [18]:
# Bonos base (Parte 2)
BONOS_PARTE2 = {
    'MUY_ALTO': 20,
    'ALTO': 50,
    'MEDIO': 50,
    'BAJO': 35,
    'MUY_BAJO': 10
}

# Ajuste por CLTV
MATRIZ_AJUSTE_CLTV = {
    'ALTO': 1.4,
    'MEDIO': 1.0,
    'BAJO': 0.6
}

def calcular_bono_parte3(row):
    bono_base = BONOS_PARTE2[row['segmento_churn']]
    multiplicador = MATRIZ_AJUSTE_CLTV[row['segmento_cltv']]
    return bono_base * multiplicador

fresh['bono_parte3'] = fresh.apply(calcular_bono_parte3, axis=1)

# Matriz de bonos
matriz_estrategias = pd.crosstab(
    fresh['segmento_churn'],
    fresh['segmento_cltv'],
    values=fresh['bono_parte3'],
    aggfunc='mean'
)

print('Matriz de Bonos Parte 3 (EUR):')
print(matriz_estrategias.round(2))

Matriz de Bonos Parte 3 (EUR):
segmento_cltv   ALTO  BAJO  MEDIO
segmento_churn                   
ALTO           70.00 30.00  50.00
BAJO           49.00 21.00  35.00
MEDIO          70.00 30.00  50.00
MUY_ALTO       28.00 12.00  20.00
MUY_BAJO       14.00  6.00  10.00


## 7. Cálculo de ROI Comparativo

In [19]:
# Parámetros comerciales
COSTE_CONTACTO = 0.5
RESPONSE_RATE = 0.10
MARGEN_BASE = 0.28

# Bonos Parte 2
fresh['bono_parte2'] = fresh['segmento_churn'].map(BONOS_PARTE2)

# ROI Parte 2
fresh['beneficio_parte2'] = (fresh['PVP'] * MARGEN_BASE - fresh['bono_parte2']) * RESPONSE_RATE - COSTE_CONTACTO
coste_total_parte2 = len(fresh) * COSTE_CONTACTO
beneficio_total_parte2 = fresh['beneficio_parte2'].sum()
roi_parte2 = (beneficio_total_parte2 / coste_total_parte2) * 100 if coste_total_parte2 > 0 else 0

# ROI Parte 3
fresh['beneficio_parte3'] = (fresh['PVP'] * MARGEN_BASE - fresh['bono_parte3']) * RESPONSE_RATE - COSTE_CONTACTO
coste_total_parte3 = len(fresh) * COSTE_CONTACTO
beneficio_total_parte3 = fresh['beneficio_parte3'].sum()
roi_parte3 = (beneficio_total_parte3 / coste_total_parte3) * 100 if coste_total_parte3 > 0 else 0

# Comparativa
resumen_roi = pd.DataFrame({
    'Metrica': ['ROI Global (%)', 'Beneficio Medio (EUR)', 'Beneficio Total (EUR)'],
    'Parte 2 (Churn)': [
        roi_parte2,
        fresh['beneficio_parte2'].mean(),
        beneficio_total_parte2
    ],
    'Parte 3 (Churn+CLTV)': [
        roi_parte3,
        fresh['beneficio_parte3'].mean(),
        beneficio_total_parte3
    ]
})

print('\nComparacion ROI Parte 2 vs Parte 3:')
print(resumen_roi.to_string(index=False))
print(f'\nMejora ROI: {(roi_parte3 - roi_parte2):+.1f}%')


Comparacion ROI Parte 2 vs Parte 3:
              Metrica  Parte 2 (Churn)  Parte 3 (Churn+CLTV)
       ROI Global (%)       130,959.29            130,977.31
Beneficio Medio (EUR)           654.80                654.89
Beneficio Total (EUR)     6,547,964.70          6,548,865.70

Mejora ROI: +18.0%


## 8. Análisis por Segmento

In [20]:
analisis = fresh.groupby('segmento_churn').agg({
    'CODE': 'count',
    'prob_churn': 'mean',
    'CLTV': 'mean',
    'bono_parte2': 'mean',
    'bono_parte3': 'mean',
    'beneficio_parte2': ['sum', 'mean'],
    'beneficio_parte3': ['sum', 'mean']
}).round(2)

analisis.columns = ['Clientes', 'Prob Churn', 'CLTV Medio', 'Bono Parte2', 'Bono Parte3', 
                     'Beneficio P2 Total', 'Beneficio P2 Media', 'Beneficio P3 Total', 'Beneficio P3 Media']

print('Analisis por Segmento de Churn:')
print(analisis)

Analisis por Segmento de Churn:
                Clientes  Prob Churn  CLTV Medio  Bono Parte2  Bono Parte3  \
segmento_churn                                                               
ALTO                5821        0.66   13,242.16        50.00        46.65   
BAJO                  24        0.28   20,233.38        35.00        41.42   
MEDIO               2163        0.55   16,327.15        50.00        54.22   
MUY_ALTO             531        0.81   12,229.20        20.00        16.87   
MUY_BAJO            1461        0.06   20,980.23        10.00        11.97   

                Beneficio P2 Total  Beneficio P2 Media  Beneficio P3 Total  \
segmento_churn                                                               
ALTO                  3,586,560.10              616.14        3,588,510.10   
BAJO                     18,815.73              783.99           18,800.33   
MEDIO                 1,585,723.08              733.11        1,584,811.08   
MUY_ALTO                357,377

## 9. Visualizaciones

In [21]:
# Scatter: Churn vs CLTV
fig1 = px.scatter(
    fresh.sample(min(2000, len(fresh))),
    x='prob_churn',
    y='CLTV',
    color='segmento_churn',
    size='bono_parte3',
    hover_data=['Modelo', 'ZONA', 'segmento_cltv'],
    title='Posicionamiento Clientes: Churn vs CLTV',
    labels={'prob_churn': 'Probabilidad Churn', 'CLTV': 'CLTV (EUR)', 'segmento_churn': 'Riesgo'},
    template='plotly_dark'
)
fig1.show()

# ROI Comparison
roi_comp = pd.DataFrame({
    'Estrategia': ['Parte 2', 'Parte 3'],
    'ROI (%)': [roi_parte2, roi_parte3]
})

fig2 = px.bar(
    roi_comp,
    x='Estrategia',
    y='ROI (%)',
    color='ROI (%)',
    title='Comparacion ROI Global',
    template='plotly_dark',
    color_continuous_scale=['red', 'yellow', 'green']
)
fig2.show()

# CLTV Distribution
fig3 = px.box(
    fresh,
    x='segmento_churn',
    y='CLTV',
    color='segmento_churn',
    title='Distribucion CLTV por Segmento',
    template='plotly_dark'
)
fig3.show()

## 10. Recomendación Final

In [22]:
print('='*80)
print('CONCLUSIONES: ANALISIS CLTV')
print('='*80)

mejora_roi = roi_parte3 - roi_parte2
mejora_beneficio = fresh['beneficio_parte3'].sum() - fresh['beneficio_parte2'].sum()

print(f'\n1. IMPACTO EN ROI:')
print(f'   Parte 2 (Churn): {roi_parte2:.1f}%')
print(f'   Parte 3 (Churn+CLTV): {roi_parte3:.1f}%')
print(f'   Mejora: {mejora_roi:+.1f}%')

print(f'\n2. IMPACTO EN BENEFICIO:')
print(f'   Parte 2: {fresh["beneficio_parte2"].sum():,.0f} EUR')
print(f'   Parte 3: {fresh["beneficio_parte3"].sum():,.0f} EUR')
print(f'   Diferencia: {mejora_beneficio:+,.0f} EUR')

if mejora_roi > 2:
    print(f'\nRECOMENDACION: Implementar Parte 3 (mejora significativa de {mejora_roi:.1f}%)')
elif mejora_roi > 0:
    print(f'\nRECOMENDACION: Considerar Parte 3 (mejora moderada de {mejora_roi:.1f}%)')
else:
    print(f'\nRECOMENDACION: Mantener Parte 2 (Parte 3 es menos eficiente)')

print('\n' + '='*80)

CONCLUSIONES: ANALISIS CLTV

1. IMPACTO EN ROI:
   Parte 2 (Churn): 130959.3%
   Parte 3 (Churn+CLTV): 130977.3%
   Mejora: +18.0%

2. IMPACTO EN BENEFICIO:
   Parte 2: 6,547,965 EUR
   Parte 3: 6,548,866 EUR
   Diferencia: +901 EUR

RECOMENDACION: Implementar Parte 3 (mejora significativa de 18.0%)

